## **[Make a Copy] InterviewOS NER Assessment — Starter Notebook**

---

### 🧭 Overview

You’ll train a **Named Entity Recognition (NER)** model that extracts product attributes such as brand, color, type, size, and material.
Your goal is to build a model that generalizes to **unseen brands and attributes**.

* Use **any Hugging Face** model (`bert-base-cased`, `roberta-base`, `deberta-v3-base`, etc.).
* Train on the provided dataset and evaluate locally on an **in-domain (ID)** test.
* Our backend will evaluate your model on a **hidden out-of-domain (OOD)** test for final scoring.
* **Model size limit:** your final zipped submission (`submission.zip`) must be **≤ 500 MB**.  
  We tested with `bert-base-cased`, which produces a model card of ≈ 190 MB.

**How to proceed**

1. **Upload** two files when prompted:
   • `product_ner_train.jsonl`
   • `product_ner_test_id.jsonl`
2. **Run** all cells to train and evaluate your model.
3. **Download** the generated `submission.zip` file and return to the InterviewOS assessment page for further instructions.

### **0. Setup**

In [ ]:
# Make sure runtime is Python 3 (default in Colab)
# Runtime → Change runtime type → Python 3 + GPU
!pip install -q transformers datasets seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### **1. Imports and Configuration**

In [ ]:
import json, time, io, shutil
from pathlib import Path
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from seqeval.metrics import classification_report, f1_score
from google.colab import files

# --------------------------------------------------------------
# Device check
# --------------------------------------------------------------
if torch.cuda.is_available():
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected.")
    print("Go to Runtime → Change runtime type → Hardware accelerator → GPU")

✅ GPU detected: Tesla T4


### **2. Load data**

In [ ]:
print("Please upload the two dataset files now:")
print("  1) product_ner_train.jsonl")
print("  2) product_ner_test_id.jsonl")

uploaded = files.upload()

required = {"product_ner_train.jsonl", "product_ner_test_id.jsonl"}
missing = required - set(uploaded.keys())
assert not missing, f"Missing required files: {missing}. Please upload both files."

def read_jsonl_bytes(b: bytes):
    rows = []
    for line in io.BytesIO(b).read().decode("utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows

train = read_jsonl_bytes(uploaded["product_ner_train.jsonl"])
test_id = read_jsonl_bytes(uploaded["product_ner_test_id.jsonl"])

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test_id": Dataset.from_list(test_id)
})

print(f"Loaded {len(train)} training and {len(test_id)} test examples.")
print("Example row:", train[0])

Please upload the two dataset files now:
  1) product_ner_train.jsonl
  2) product_ner_test_id.jsonl


Saving product_ner_train.jsonl to product_ner_train.jsonl
Saving product_ner_test_id.jsonl to product_ner_test_id.jsonl
Loaded 400 training and 100 test examples.
Example row: {'tokens': ['west elm', 'Wooden', 'desk'], 'ner_tags': ['B-BRAND', 'B-MATERIAL', 'B-TYPE']}


### **3. Label mapping**

In [ ]:
label_list = sorted({t for ex in (train + test_id) for t in ex["ner_tags"]})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("Labels:", label_list)

Labels: ['B-BRAND', 'B-COLOR', 'B-GENDER', 'B-MATERIAL', 'B-SIZE', 'B-TYPE', 'I-TYPE']


### **4. Tokenization (TODO)**

In [ ]:
# TODO: implement tokenization and label alignment
# Tips:
#  - Use tokenizer(..., is_split_into_words=True)
#  - Assign -100 to ignored positions (subwords / special tokens)
#  - Return a dict that includes "labels" aligned to tokens

def tokenize_and_align_labels(examples):
    # raise NotImplementedError("Implement tokenization logic here.")
    pass

### **5. Model and tokenizer**

In [ ]:
# TODO: choose your base model and tokenizer
# Example options: "bert-base-cased", "roberta-base", "distilbert-base-uncased"
MODEL_NAME = "???"
TOKENIZER_KWARGS = {}  # e.g., {"add_prefix_space": True} if using RoBERTa

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **TOKENIZER_KWARGS)

# TODO: apply your tokenization function to dataset
# tokenized = dataset.map(tokenize_and_align_labels, batched=True)

# TODO: initialize your model for token classification.
# Please name it as "model" for future blocks of submission download.
# model = AutoModelForTokenClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=len(label_list),
#     id2label=id2label,
#     label2id=label2id,
# )

### **6. Training**

In [ ]:
# TODO: configure your training arguments
# Decide on learning rate, epochs, and batch size
args = TrainingArguments(
    output_dir="./ner_results",
    learning_rate=???,
    per_device_train_batch_size=???, # ≤16 recommended
    num_train_epochs=???, # ≤4 recommended
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
)

# TODO: create Trainer and train your model
# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized["train"],
#     tokenizer=tokenizer,
#     data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
# )
# trainer.train()

### **7. Local evaluation (ID test)**

In [ ]:
# TODO: run prediction and compute metrics on the test set
# You may use trainer.predict(...) or write your own loop.

# Example outline:
# preds_logits, labels, _ = trainer.predict(tokenized["test_id"])
# preds = preds_logits.argmax(axis=-1)
# true_labels, pred_labels = ...
# print(classification_report(true_labels, pred_labels))

# **8. Save submission**

In [ ]:
SUBMIT_DIR = Path("submission")
SUBMIT_DIR.mkdir(exist_ok=True)

# Move to float16 to reduce size
model = model.to(torch.float16)

# Save compact model/tokenizer
model.save_pretrained(SUBMIT_DIR / "model", safe_serialization=True)
tokenizer.save_pretrained(SUBMIT_DIR / "model")

# Zip and download
shutil.make_archive("submission", "zip", SUBMIT_DIR)
files.download("model_card.zip")

### **9. (Optional) Latency check**

In [ ]:
sample = " ".join(test_id[0]["tokens"])
inputs = tokenizer(sample, return_tensors="pt", padding=True, truncation=True)

# Move everything to the same device
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Warm-up pass
with torch.no_grad():
    _ = model(**inputs)
if device == "cuda":
    torch.cuda.synchronize()

# Timed forward pass
t0 = time.time()
with torch.no_grad():
    _ = model(**inputs)
if device == "cuda":
    torch.cuda.synchronize()
print("Single forward latency (s):", round(time.time() - t0, 4))

Single forward latency (s): 0.0405


### **10. Checklist**

* [ ] Dataset uploaded successfully
* [ ] Tokenization and alignment implemented
* [ ] Model trained within ~4 epochs, batch ≤ 16
* [ ] Evaluated on visible test (ID)
* [ ] Downloaded `submission.zip`
* [ ] Notebook runs end-to-end